In [1208]:
import re
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline

In [1209]:

def parse_experiment_data(file_path):
    """
    Parses the experiment data file and extracts the experiment number, date, 
    and time series with values for each sample.
    
    Args:
        file_path (str): Path to the experiment data file.
    
    Returns:
        dict: Parsed data in the format:
            {
                "num": experiment_number,
                "date": date_collected,
                "samples": {
                    "Sample001": {"time": [times], "values": [values]},
                    ...
                }
            }
    """
    parsed_data = {}

    try:
        with open(file_path, "r") as file:
            lines = file.readlines()
        
        # Extract the experiment number
        experiment_line = lines[0].strip()
        match = re.search(r"(?:rate|mads_t)(\d+)\.rre", experiment_line, re.IGNORECASE)
        if match:
            parsed_data["num"] = int(match.group(1))
        else:
            parsed_data["num"] = None
            print("Warning: Experiment number not found.")

        # Extract the date
        if len(lines) > 5:
            date_line = lines[5].strip().split('\t')
            if len(date_line) > 1:
                parsed_data["date"] = date_line[1]
            else:
                parsed_data["date"] = None
                print("Warning: Date not found in the expected format.")
        else:
            parsed_data["date"] = None
            print("Warning: Date line is missing.")

        # Extract sample names
        if len(lines) > 2:
            sample_names = lines[2].strip().split('\t')[1::2]
            parsed_data["samples"] = {sample: {"time": [], "values": []} for sample in sample_names}
        else:
            parsed_data["samples"] = {}
            print("Warning: Sample names line is missing.")

        # Locate the start of the data section
        for i, line in enumerate(lines):
            if line.lower().startswith("ss.sss"):
                data_start_line = i + 1
                break
        else:
            raise ValueError("Data section not found in the file.")
        
        # Parse the data
        for line in lines[data_start_line:]:
            values = line.strip().split('\t')
            if not values or len(values) < 2:  # Skip empty or incomplete lines
                continue
            try:
                time = float(values[0])  # Extract time from the first column
            except ValueError:
                print(f"Warning: Invalid time value in line: {line}")
                continue

            for i, sample in enumerate(parsed_data["samples"]):
                value_index = 2 * i + 1  # Locate the corresponding value column
                if value_index < len(values):  # Check if the value exists
                    try:
                        value = float(values[value_index])
                        parsed_data["samples"][sample]["time"].append(time)
                        parsed_data["samples"][sample]["values"].append(value)
                    except ValueError:
                        print(f"Warning: Invalid value for {sample} in line: {line}")
                        continue

    except FileNotFoundError:
        print(f"Error: File not found at {file_path}")
        return None
    except Exception as e:
        print(f"Error: An unexpected error occurred: {e}")
        return None

    return parsed_data


In [1210]:

def resample_time_series_with_offset(time, values, interval=60, plot=True):
    """
    Resamples a time series to a uniform time step using cubic spline interpolation,
    adjusts the values to start at 0, and plots the original data, fitted spline,
    and resampled points. If the second time value equals the interval, assumes the
    series already has the correct time step and skips resampling.
    
    Args:
        time (list or array): Original time points.
        values (list or array): Original values corresponding to time points.
        interval (int): Desired uniform time step in seconds.
        plot (bool): Whether to plot the results.
        
    Returns:
        dict: Resampled time series with keys 'Time' and 'Values'.
    """
    # Convert to numpy arrays for compatibility
    time = np.array(time)
    values = np.array(values)
    # Check if the time step matches the desired interval
    if len(time) > 1 and np.isclose(time[1] - time[0], interval):
        # Apply offset directly without resampling
        value_offset = values[0]
        adjusted_values = values - value_offset
        
        if plot:
            # Plot the results
            plt.figure(figsize=(10, 6))
            plt.plot(time, adjusted_values, 'o', label="Original Data (Offset Applied)", markersize=8)
            plt.xlabel("Time (seconds)")
            plt.ylabel("Values (Offset)")
            plt.title("Time Series with Offset Applied (No Resampling Needed)")
            plt.legend()
            plt.grid(True)
            plt.show()
        
        return {"time": time, "values": adjusted_values}
    
    # Generate new time points at uniform intervals starting from the first time
    start_time = time[0]
    new_time = np.arange(start_time, time[-1] + interval, interval)
    
    # Fit a cubic spline to the original data
    spline = CubicSpline(time, values)
    
    # Evaluate the spline at the new time points
    new_values = spline(new_time)
    
    # Apply the offset (subtract the value at the first new time point)
    value_offset = new_values[0]
    new_values -= value_offset
    
    if plot:
        # Plot the results
        plt.figure(figsize=(10, 6))
        plt.plot(time, values, 'o', label="Original Data", markersize=8)
        plt.plot(new_time, new_values, 'x', label="Resampled Points (Offset)", markersize=8)
        plt.xlabel("Time (seconds)")
        plt.ylabel("Values (Offset)")
        plt.title("Time Series Resampling with Spline Interpolation (Offset Applied)")
        plt.legend()
        plt.grid(True)
        plt.show()
    
    # Return the resampled time and values
    return {"time": new_time, "values": new_values}


In [1211]:

def find_and_parse_experiment_file(experiment_number, directory, sheet_name=None):
    """
    Finds the associated .xls file for a given experiment number by searching recursively
    through subdirectories starting from the provided path and parses the specified sheet.
    If the primary method fails, uses the #num pattern as a fallback.
    
    Args:
        experiment_number (int): The experiment number to search for (e.g., 26 for mads_t026).
        directory (str): The directory where the search starts.
        sheet_name (str, optional): Name of the sheet to parse. If None, the first sheet is used.
    
    Returns:
        tuple: A tuple containing the filename and a DataFrame with the parsed data, 
               or None if no file is found.
    """
    # Primary pattern for mads_t<num> or rate<num>
    primary_pattern = re.compile(rf"(mads_t|rate){experiment_number:03d}.*\.xls", re.IGNORECASE)
    # Fallback pattern for #<num>
    fallback_pattern = re.compile(rf"#\s*{experiment_number}\.xls", re.IGNORECASE)
    
    # Search recursively for files
    for file in Path(directory).rglob("*.xls"):
        # Check for primary pattern first
        if primary_pattern.search(file.name):
            try:
                return parse_file(file, sheet_name)
            except Exception as e:
                print(f"Error parsing file {file}: {e}")
                return None
    
    # If no file is found using the primary pattern, use the fallback pattern
    for file in Path(directory).rglob("*.xls"):
        if fallback_pattern.search(file.name):
            try:
                return parse_file(file, sheet_name)
            except Exception as e:
                print(f"Error parsing file {file}: {e}")
                return None
    
    # If no file is found
    print(f"No file found for experiment number {experiment_number}.")
    return None


def parse_file(file, sheet_name=None):
    """
    Parses the specified .xls file.
    
    Args:
        file (Path): Path to the file.
        sheet_name (str, optional): Name of the sheet to parse. If None, the first sheet is used.
    
    Returns:
        tuple: A tuple containing the filename and a DataFrame with the parsed data.
    """
    print(f"File found: {file.name}")
    
    # Read the .xls file without headers
    if sheet_name:
        # Read the specified sheet
        data = pd.read_excel(file, sheet_name=sheet_name, header=None)
    else:
        # Read the entire file and default to the first sheet
        df = pd.read_excel(file, sheet_name=None, header=None)  # Load all sheets
        print(f"Available sheets: {list(df.keys())}")
        first_sheet_name = list(df.keys())[0]
        data = df[first_sheet_name]

    # Drop empty rows and columns
    # data = data.dropna(how="all", axis=1)  # Drop empty columns
    # data = data.dropna(how="all", axis=0)  # Drop empty rows
    
    return file.name, data




In [1212]:
def find_header_row(data, search_strings=None):
    """
    Locates the header row in a given DataFrame by searching for any of the specified strings
    with exact matches, disregarding capitalization.

    Args:
        data (pd.DataFrame): The DataFrame to search through.
        search_strings (list of str): List of strings to search for in the cells.

    Returns:
        int: Index of the header row, or None if no match is found.
    """
    if search_strings is None:
        search_strings = ["[Enz]", "Kuv.", "[Enz] mmol/l", "sub"]  # Default strings to search for

    # Convert the list of search strings to lowercase for case-insensitive matching
    search_strings = [s.lower() for s in search_strings]

    # Iterate through rows of the DataFrame
    for i, row in data.iterrows():
        # Check if any cell matches any search string exactly (case-insensitive)
        if any(str(cell).strip().lower() in search_strings for cell in row):
            return i
    
    # If no match is found
    print(f"Header row containing one of {search_strings} not found.")
    return None


In [1213]:
def find_numeric_values_below_header(data, header_row, sample_num):
    """
    Searches the header row for specific strings (case-insensitive), and for each match,
    finds the first 'sample_num' numeric values below the header row in the corresponding columns.
    Maps the found values to standardized keys: [enz], [h2o2], and [sub].
    If there are multiple matches for a string in the header row, selects values from the column 
    with the largest first numeric value. The values are rounded to 3 decimal places.
    If any standardized key remains None, sets it to a list of zeros.
    
    Args:
        data (pd.DataFrame): The DataFrame containing the data.
        header_row (int): The index of the header row.
        sample_num (int): The number of numeric values to include (default is 1).
     
    Returns:
        dict: A dictionary with standardized keys ([enz], [h2o2], [sub]) and lists of rounded
              numeric values as values.
    """
    results = {"[enz]": None, "[h2o2]": None, "[sub]": None}
    
    # Mapping of standardized keys to search terms
    search_mapping = {
        "[enz]": ["[enz]", "enz"],
        "[h2o2]": ["[h2o2]", "h2o2"],
        "[sub]": ["[sub]", "sub"]
    }
    
    # Extract the header row
    header = data.iloc[header_row]
    
    # Iterate over the columns in the header
    for col in data.columns:
        cell_value = str(header[col]).lower()  # Convert to lowercase for case-insensitive comparison
        
        # Check if the cell contains any of the search terms
        for standardized_key, search_terms in search_mapping.items():
            if any(term in cell_value for term in search_terms):
                # Find numeric values below the header
                numeric_values = []
                for value in data[col].iloc[header_row + 1:]:
                    if isinstance(value, (int, float)):
                        numeric_values.append(round(value, 3))
                    if len(numeric_values) == sample_num:
                        break
                
                if numeric_values:
                    # Store the values, prioritizing the column with the largest first value
                    if results[standardized_key] is None or numeric_values[0] > results[standardized_key][0]:
                        results[standardized_key] = numeric_values
    
    # Set any None results to a list of zeros
    for key in results:
        if results[key] is None:
            results[key] = [0] * sample_num
    
    return results


In [1214]:
def find_pH_value_in_range(data, row_range, col_range, filename=None):
    """
    Finds the pH value located in a cell to the immediate right of a cell containing the string "pH",
    within a specified range of rows and columns. If not found in the DataFrame, attempts to parse the 
    pH value from the filename using the patterns "pH=XX.XX" or "pH_XX,XX".
    
    Args:
        data (pd.DataFrame): The DataFrame containing the data.
        row_range (tuple): A tuple specifying the start and end row indices (inclusive).
        col_range (tuple): A tuple specifying the start and end column indices (inclusive).
        filename (str, optional): The filename to parse the pH from as a fallback.
    
    Returns:
        float: The pH value if found and numeric, or None if not found or is NaN.
    """
    # Extract the subrange of the DataFrame to search
    start_row, end_row = row_range
    start_col, end_col = col_range
    search_area = data.iloc[start_row:end_row + 1, start_col:end_col + 1]
    
    # Iterate through the specified range
    for row_index in range(search_area.shape[0]):
        for col_index in range(search_area.shape[1] - 1):  # Stop before the last column
            cell = search_area.iloc[row_index, col_index]
            # Match "pH" as a whole word
            if isinstance(cell, str) and re.fullmatch(r"\bpH\b", cell, re.IGNORECASE):
                # Check the cell to the right
                next_cell = search_area.iloc[row_index, col_index + 1]
                
                # Handle numeric types directly
                if isinstance(next_cell, float) and np.isnan(next_cell):
                    print("Adjacent cell contains NaN.")
                    return None
                if isinstance(next_cell, (int, float)):
                    return round(next_cell, 2)
                
                print(f"Adjacent value is not numeric: {next_cell}")
                return None
    
    # If 'pH' is not found, attempt to parse from the filename
    if filename:
        # First pattern: pH=XX.XX
        match = re.search(r"pH=([\d.]+)", filename, re.IGNORECASE)
        if match:
            return round(float(match.group(1)), 2)
        
        # Second pattern: pH_XX,XX (comma as decimal separator)
        match = re.search(r"pH_([\d,]+)", filename, re.IGNORECASE)
        if match:
            pH_value = match.group(1).replace(",", ".")  # Convert comma to dot for float conversion
            return round(float(pH_value), 3)
        
        print("pH value not found in filename.")
    
    # If neither DataFrame nor filename contains pH information
    print("pH string not found in the specified range or filename.")
    return None


In [1215]:
def find_temperature_value_in_range(data, row_range, col_range, filename=None):
    """
    Finds the temperature value located in a cell to the immediate right of a cell containing the string "T",
    within a specified range of rows and columns. If not found, searches for any cell in the range containing
    a string like "XX C" and extracts the numeric temperature value. If still not found, attempts to parse the
    temperature from the filename using the pattern "t=XX".
    
    Args:
        data (pd.DataFrame): The DataFrame containing the data.
        row_range (tuple): A tuple specifying the start and end row indices (inclusive).
        col_range (tuple): A tuple specifying the start and end column indices (inclusive).
        filename (str, optional): The filename to parse the temperature from as a fallback.
    
    Returns:
        float: The temperature value if found and numeric, or None if not found.
    """
    # Extract the subrange of the DataFrame to search
    start_row, end_row = row_range
    start_col, end_col = col_range
    search_area = data.iloc[start_row:end_row + 1, start_col:end_col + 1]
    
    # First attempt: Find "T" and get the adjacent cell
    for row_index in range(search_area.shape[0]):
        for col_index in range(search_area.shape[1] - 1):  # Stop before the last column
            cell = search_area.iloc[row_index, col_index]
            if isinstance(cell, str) and re.fullmatch(r"\bT\b", cell, re.IGNORECASE):
                # Check the cell to the right
                next_cell = search_area.iloc[row_index, col_index + 1]
                
                # Handle numeric types directly
                if isinstance(next_cell, (int, float)):
                    return round(next_cell, 3)
                
                print(f"Adjacent value is not numeric: {next_cell}")
                return None
    
    # Fallback: Search for cells with "XX C" pattern directly in the range
    for row_index in range(search_area.shape[0]):
        for col_index in range(search_area.shape[1]):
            cell = search_area.iloc[row_index, col_index]
            if isinstance(cell, str):
                match = re.search(r"(\d+(\.\d+)?)\s*C", cell, re.IGNORECASE)
                if match:
                    return int(match.group(1))
    
    # Fallback: Attempt to parse from the filename
    if filename:
        match = re.search(r"t=(\d+)", filename, re.IGNORECASE)
        if match:
            return int(match.group(1))
    
    # If no temperature information is found
    print("Temperature value not found in the specified range or filename.")
    return None

In [1216]:
def find_buffer_type(data, row_range, col_range, filename=None, buffer_map=None):
    """
    Finds the buffer type within a specified range of rows and columns in a DataFrame.
    If not found in the DataFrame, attempts to parse the buffer type from the filename.
    Maps found buffer types to standardized names using a buffer_map.
    
    Args:
        data (pd.DataFrame): The DataFrame containing the data.
        row_range (tuple): A tuple specifying the start and end row indices (inclusive).
        col_range (tuple): A tuple specifying the start and end column indices (inclusive).
        filename (str, optional): The filename to parse the buffer type from as a fallback.
        buffer_map (dict, optional): Mapping of buffer type keywords to standardized names.
    
    Returns:
        str: The standardized buffer type if found, or None if not found.
    """
    if buffer_map is None:
        buffer_map = {
            "boric": "Boric Acid",
            "pyrophosphate": "Pyrophosphate",
            "phosphat": "Phosphate",
            "phosphate": "Phosphate",
            "Na4P2O7*10H2O" : "Pyrophosphate",
            "NaH2PO4*2H2O" : "Pyrophosphate",
            "carbonate": "Carbonate"
        }
    
    # Create a set of all keywords to search for (case-insensitive)
    keywords = {key.lower(): value for key, value in buffer_map.items()}
    
    # Extract the subrange of the DataFrame to search
    start_row, end_row = row_range
    start_col, end_col = col_range
    search_area = data.iloc[start_row:end_row + 1, start_col:end_col + 1]
    
    # Search the DataFrame for buffer types
    for row_index in range(search_area.shape[0]):
        for col_index in range(search_area.shape[1]):
            cell = search_area.iloc[row_index, col_index]
            if isinstance(cell, str):
                cell_lower = cell.lower()
                for keyword, standardized_name in keywords.items():
                    if keyword in cell_lower:
                        return standardized_name
    
    # If no buffer type is found, attempt to parse from the filename
    if filename:
        filename_lower = filename.lower()
        for keyword, standardized_name in keywords.items():
            if keyword in filename_lower:
                return standardized_name
    
    # If neither DataFrame nor filename contains buffer type information
    print("Buffer type not found in the specified range or filename.")
    return None


In [1217]:
def find_substrate_type(data, row_range, col_range, filename=None):
    """
    Finds the substrate type within a specified range of rows and columns in a DataFrame.
    If not found in the DataFrame, attempts to parse the substrate type from the filename.
    Maps recognized substrate types to properly formatted values.
    
    Args:
        data (pd.DataFrame): The DataFrame containing the data.
        row_range (tuple): A tuple specifying the start and end row indices (inclusive).
        col_range (tuple): A tuple specifying the start and end column indices (inclusive).
        filename (str, optional): The filename to parse the substrate type from as a fallback.
    
    Returns:
        str: The properly formatted substrate type if found, or an empty string if not found.
    """
    # Substrate types and their standardized mappings
    substrate_mapping = {
        "bnoh": "BnOH",
        "benzylalkohol": "BnOH",
        "4ome-bnoh": "4OMe-BnOH",
        "4-meo-bnoh": "4OMe-BnOH",
        "4-methoxy-benzylalkohol": "4OMe-BnOH"
    }
    
    # Convert mapping keys to lowercase for case-insensitive comparison
    search_terms = {key.lower(): value for key, value in substrate_mapping.items()}
    
    # Extract the subrange of the DataFrame to search
    start_row, end_row = row_range
    start_col, end_col = col_range
    search_area = data.iloc[start_row:end_row + 1, start_col:end_col + 1]
    
    # Search the DataFrame for substrate types
    for row_index in range(search_area.shape[0]):
        for col_index in range(search_area.shape[1]):
            cell = search_area.iloc[row_index, col_index]
            if isinstance(cell, str):
                cell_lower = cell.lower()
                for term, standardized in search_terms.items():
                    # Match exact, start-of-string followed by whitespace, or whitespace-enclosed matches
                    if re.search(rf"(^|\s){re.escape(term)}(\s|$)", cell_lower):
                        return standardized
    
    # If no substrate type is found, attempt to parse from the filename
    if filename:
        filename_lower = filename.lower()
        for term, standardized in search_terms.items():
            if re.search(rf"(^|\s){re.escape(term)}(\s|$)", filename_lower):
                return standardized
    
    # If neither DataFrame nor filename contains substrate type information
    print("Substrate type not found in the specified range or filename.")
    return ""


In [1221]:
file_path = "data/data/data145.txt"  # Replace with your actual file path
row_range = (0, 70)  # Search rows 0 to 3
col_range = (0, 20)  # Search columns 0 to 2

# Parse the file
parsed_data = parse_experiment_data(file_path)
resampled = resample_time_series_with_offset(parsed_data['samples']['Sample001']['time'], parsed_data['samples']['Sample001']['values'], interval=60, plot=False)
file_name, experiment_data = find_and_parse_experiment_file(experiment_number=parsed_data["num"], directory='data', sheet_name='Sheet1')
header_row = find_header_row(experiment_data)
results = find_numeric_values_below_header(experiment_data, header_row, sample_num=len(parsed_data['samples']))
pH_value = find_pH_value_in_range(experiment_data, row_range, col_range, filename=file_name)
temperature = find_temperature_value_in_range(experiment_data, row_range, col_range, filename=file_name)
buffer_type = find_buffer_type(experiment_data, row_range, col_range, filename=file_name)
substrate_type = find_substrate_type(experiment_data, row_range, col_range, filename=file_name)

# Display result
print("Results:", results)
print("Substrate Type:", substrate_type)
print("Buffer Type:", buffer_type)
print("pH Value:", pH_value)
print("Temperature Value:", temperature)


File found: BnOH_pH_9,04_H2O2_35,24_#145.xls
Results: {'[enz]': [0.021, 0.021, 0.021, 0.021, 0.021, 0.021, 0.021], '[h2o2]': [35.244, 35.244, 35.244, 35.244, 24.964, 14.685, 5.14], '[sub]': [10.816, 3.028, 0.865, 0.216, 10.816, 10.816, 10.816]}
Substrate Type: BnOH
Buffer Type: Pyrophosphate
pH Value: 9.04
Temperature Value: 25
